# CNN Fundamentals

This notebook builds intuition for Convolutional Neural Networks (CNNs) without requiring deep learning frameworks:
1. CNN architecture explained with diagrams.
2. Manual convolution operation with NumPy.
3. How different filters detect edges, corners, and textures.
4. Visualize filter outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle
from scipy import ndimage

plt.rcParams['figure.figsize'] = (12, 6)
print('Imports ready.')

## 1. CNN Architecture Diagram

A typical CNN consists of:
- **Convolutional layers**: Apply learnable filters to detect features.
- **Pooling layers**: Downsample feature maps to reduce dimensionality.
- **Fully connected layers**: Classify based on extracted features.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
ax.set_xlim(0, 18)
ax.set_ylim(0, 7)
ax.axis('off')

# Define layers
layers = [
    {'name': 'Input\nImage\n28x28x1', 'x': 1, 'w': 1.2, 'h': 4.0, 'color': '#3498db'},
    {'name': 'Conv1\n5x5, 32\nfilters', 'x': 3.5, 'w': 1.0, 'h': 3.5, 'color': '#e74c3c'},
    {'name': 'Pool1\n2x2\nmax', 'x': 5.5, 'w': 0.8, 'h': 2.8, 'color': '#f39c12'},
    {'name': 'Conv2\n3x3, 64\nfilters', 'x': 7.5, 'w': 1.0, 'h': 2.5, 'color': '#e74c3c'},
    {'name': 'Pool2\n2x2\nmax', 'x': 9.5, 'w': 0.8, 'h': 2.0, 'color': '#f39c12'},
    {'name': 'Flatten', 'x': 11.5, 'w': 0.6, 'h': 1.5, 'color': '#9b59b6'},
    {'name': 'FC\n128\nneurons', 'x': 13.5, 'w': 0.8, 'h': 2.5, 'color': '#2ecc71'},
    {'name': 'Output\n10\nclasses', 'x': 15.5, 'w': 0.8, 'h': 1.8, 'color': '#1abc9c'},
]

for layer in layers:
    rect = Rectangle((layer['x'] - layer['w']/2, 3.5 - layer['h']/2),
                      layer['w'], layer['h'],
                      facecolor=layer['color'], edgecolor='black',
                      linewidth=1.5, alpha=0.8)
    ax.add_patch(rect)
    ax.text(layer['x'], 3.5, layer['name'], ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

# Arrows between layers
for i in range(len(layers) - 1):
    x_start = layers[i]['x'] + layers[i]['w']/2 + 0.1
    x_end = layers[i+1]['x'] - layers[i+1]['w']/2 - 0.1
    ax.annotate('', xy=(x_end, 3.5), xytext=(x_start, 3.5),
               arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Labels
ax.text(4.5, 6.2, 'Feature Extraction', ha='center', fontsize=13,
        fontweight='bold', color='#e74c3c',
        bbox=dict(boxstyle='round', facecolor='#fadbd8', edgecolor='#e74c3c'))
ax.annotate('', xy=(2.5, 5.8), xytext=(6.5, 5.8),
           arrowprops=dict(arrowstyle='<->', color='#e74c3c', lw=2))

ax.text(14.5, 6.2, 'Classification', ha='center', fontsize=13,
        fontweight='bold', color='#2ecc71',
        bbox=dict(boxstyle='round', facecolor='#d5f5e3', edgecolor='#2ecc71'))
ax.annotate('', xy=(13, 5.8), xytext=(16, 5.8),
           arrowprops=dict(arrowstyle='<->', color='#2ecc71', lw=2))

ax.set_title('Typical CNN Architecture', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Understanding Convolution

Convolution slides a small filter (kernel) over the image, computing a dot product at each position.

In [ ]:
# Create a small image for demonstration
small_img = np.array([
    [0,  0,  0,  0,  0,  0,  0],
    [0,  0,  50, 50, 50, 0,  0],
    [0,  0,  50, 100,50, 0,  0],
    [0,  50, 100,200,100,50, 0],
    [0,  0,  50, 100,50, 0,  0],
    [0,  0,  50, 50, 50, 0,  0],
    [0,  0,  0,  0,  0,  0,  0],
], dtype=np.float64)

kernel = np.array([[-1, -1, -1],
                    [-1,  8, -1],
                    [-1, -1, -1]], dtype=np.float64)

print('Input image (7x7):')
print(small_img.astype(int))
print('\nKernel (3x3):')
print(kernel.astype(int))

In [ ]:
def convolve2d_manual(image, kernel):
    """Perform 2D convolution manually (valid mode)."""
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih - kh + 1, iw - kw + 1
    output = np.zeros((oh, ow))
    
    for i in range(oh):
        for j in range(ow):
            region = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * kernel)
    
    return output

# Manual convolution
output = convolve2d_manual(small_img, kernel)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(small_img, cmap='gray')
axes[0].set_title('Input Image (7x7)')
for i in range(7):
    for j in range(7):
        axes[0].text(j, i, f'{small_img[i,j]:.0f}', ha='center', va='center',
                    fontsize=8, color='red' if small_img[i,j] > 100 else 'blue')

axes[1].imshow(kernel, cmap='RdBu_r')
axes[1].set_title('Kernel (3x3)')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel[i,j]:.0f}', ha='center', va='center',
                    fontsize=12, fontweight='bold')

axes[2].imshow(output, cmap='RdBu_r')
axes[2].set_title('Output (5x5)')
for i in range(output.shape[0]):
    for j in range(output.shape[1]):
        axes[2].text(j, i, f'{output[i,j]:.0f}', ha='center', va='center',
                    fontsize=8)

plt.suptitle('Manual Convolution Operation', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Different Filters for Different Features

Different kernels detect different types of features in images.

In [ ]:
# Create a richer test image
def create_test_scene(size=128):
    """Create a synthetic scene with edges, corners, and textures."""
    img = np.ones((size, size)) * 180
    
    # Rectangle (edges)
    img[20:60, 20:80] = 40
    
    # Circle
    y, x = np.ogrid[:size, :size]
    circle = (x - 100)**2 + (y - 40)**2 < 25**2
    img[circle] = 60
    
    # Checkerboard texture
    block = 8
    for i in range(70, 120, block):
        for j in range(10, 60, block):
            if ((i-70)//block + (j-10)//block) % 2 == 0:
                img[i:min(i+block,size), j:min(j+block,size)] = 40
    
    # Diagonal lines
    for k in range(50):
        row = 70 + k
        col = 70 + k
        if row < size and col < size:
            img[row, max(0,col-1):min(size,col+2)] = 30
    
    # Gradient region
    img[80:120, 80:120] = np.tile(np.linspace(20, 220, 40), (40, 1))
    
    return img

test_scene = create_test_scene(128)
plt.figure(figsize=(6, 6))
plt.imshow(test_scene, cmap='gray')
plt.title('Synthetic Test Scene')
plt.axis('off')
plt.show()

In [ ]:
# Define a collection of filters
filters = {
    'Horizontal Edge': np.array([[-1, -1, -1],
                                  [ 0,  0,  0],
                                  [ 1,  1,  1]], dtype=np.float64),
    
    'Vertical Edge':   np.array([[-1, 0, 1],
                                  [-1, 0, 1],
                                  [-1, 0, 1]], dtype=np.float64),
    
    'Diagonal Edge':   np.array([[ 0,  1,  1],
                                  [-1,  0,  1],
                                  [-1, -1,  0]], dtype=np.float64),
    
    'Sharpen':         np.array([[ 0, -1,  0],
                                  [-1,  5, -1],
                                  [ 0, -1,  0]], dtype=np.float64),
    
    'Gaussian Blur':   np.array([[1, 2, 1],
                                  [2, 4, 2],
                                  [1, 2, 1]], dtype=np.float64) / 16.0,
    
    'Laplacian':       np.array([[ 0,  1,  0],
                                  [ 1, -4,  1],
                                  [ 0,  1,  0]], dtype=np.float64),
    
    'Emboss':          np.array([[-2, -1, 0],
                                  [-1,  1, 1],
                                  [ 0,  1, 2]], dtype=np.float64),
    
    'Corner Detect':   np.array([[-1, -1, -1],
                                  [-1,  8, -1],
                                  [-1, -1, -1]], dtype=np.float64),
}

In [ ]:
# Apply all filters and visualize
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for idx, (name, kernel) in enumerate(filters.items()):
    row, col = idx // 4, idx % 4
    filtered = ndimage.convolve(test_scene, kernel)
    axes[row, col].imshow(filtered, cmap='gray')
    axes[row, col].set_title(name, fontsize=11)
    axes[row, col].axis('off')

plt.suptitle('Different Filters Detect Different Features', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Visualize Filter Kernels

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, (name, kernel) in enumerate(filters.items()):
    row, col = idx // 4, idx % 4
    ax = axes[row, col]
    im = ax.imshow(kernel, cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_title(name, fontsize=10)
    # Annotate values
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            val = kernel[i, j]
            ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                   fontsize=10, fontweight='bold',
                   color='white' if abs(val) > 2 else 'black')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Filter Kernels Visualized', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Max Pooling Demonstration

Pooling reduces spatial dimensions by taking the maximum (or average) value in each window.

In [ ]:
def max_pool_2d(image, pool_size=2):
    """Apply max pooling with given pool size."""
    h, w = image.shape
    oh, ow = h // pool_size, w // pool_size
    output = np.zeros((oh, ow))
    
    for i in range(oh):
        for j in range(ow):
            region = image[i*pool_size:(i+1)*pool_size,
                          j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(region)
    
    return output

# Demonstrate on a small example
small_feature = np.array([
    [1,  3,  2,  4],
    [5,  6,  7,  8],
    [3,  2,  1,  0],
    [1,  4,  3,  2],
], dtype=np.float64)

pooled = max_pool_2d(small_feature, pool_size=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(small_feature, cmap='YlOrRd')
ax1.set_title('Input Feature Map (4x4)')
for i in range(4):
    for j in range(4):
        ax1.text(j, i, f'{small_feature[i,j]:.0f}', ha='center', va='center', fontsize=14)
# Draw pool regions
for i in range(2):
    for j in range(2):
        rect = Rectangle((j*2-0.5, i*2-0.5), 2, 2, linewidth=3,
                         edgecolor='blue', facecolor='none')
        ax1.add_patch(rect)

ax2.imshow(pooled, cmap='YlOrRd')
ax2.set_title('After 2x2 Max Pooling (2x2)')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{pooled[i,j]:.0f}', ha='center', va='center',
                fontsize=18, fontweight='bold')

plt.suptitle('Max Pooling Operation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Apply to real feature maps
# First convolve, then pool
edge_map = ndimage.convolve(test_scene, filters['Horizontal Edge'])
pooled_1x = max_pool_2d(np.abs(edge_map), pool_size=2)
pooled_2x = max_pool_2d(pooled_1x, pool_size=2)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(test_scene, cmap='gray')
axes[0].set_title(f'Original ({test_scene.shape[0]}x{test_scene.shape[1]})')
axes[0].axis('off')

axes[1].imshow(np.abs(edge_map), cmap='gray')
axes[1].set_title(f'Edge Feature Map ({edge_map.shape[0]}x{edge_map.shape[1]})')
axes[1].axis('off')

axes[2].imshow(pooled_1x, cmap='gray')
axes[2].set_title(f'After Pool 1 ({pooled_1x.shape[0]}x{pooled_1x.shape[1]})')
axes[2].axis('off')

axes[3].imshow(pooled_2x, cmap='gray')
axes[3].set_title(f'After Pool 2 ({pooled_2x.shape[0]}x{pooled_2x.shape[1]})')
axes[3].axis('off')

plt.suptitle('Progressive Downsampling Through Pooling', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Simulating Multiple Filter Outputs (Feature Maps)

In a CNN, a convolutional layer applies many filters simultaneously, producing multiple feature maps.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Row 0: Original + 3 edge filters
axes[0, 0].imshow(test_scene, cmap='gray')
axes[0, 0].set_title('Input Image', fontsize=11)
axes[0, 0].axis('off')

edge_filters = ['Horizontal Edge', 'Vertical Edge', 'Diagonal Edge']
for i, name in enumerate(edge_filters):
    output = ndimage.convolve(test_scene, filters[name])
    axes[0, i+1].imshow(output, cmap='RdBu_r')
    axes[0, i+1].set_title(f'Filter: {name}', fontsize=10)
    axes[0, i+1].axis('off')

# Row 1: Absolute values (feature activations)
axes[1, 0].text(0.5, 0.5, 'Feature\nActivations\n(|output|)', transform=axes[1, 0].transAxes,
               ha='center', va='center', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

for i, name in enumerate(edge_filters):
    output = ndimage.convolve(test_scene, filters[name])
    axes[1, i+1].imshow(np.abs(output), cmap='hot')
    axes[1, i+1].set_title(f'|{name}|', fontsize=10)
    axes[1, i+1].axis('off')

# Row 2: After max pooling
axes[2, 0].text(0.5, 0.5, 'After\nMax Pooling\n(2x2)', transform=axes[2, 0].transAxes,
               ha='center', va='center', fontsize=12, fontweight='bold')
axes[2, 0].axis('off')

for i, name in enumerate(edge_filters):
    output = ndimage.convolve(test_scene, filters[name])
    pooled = max_pool_2d(np.abs(output), pool_size=2)
    axes[2, i+1].imshow(pooled, cmap='hot')
    axes[2, i+1].set_title(f'Pooled {name}', fontsize=10)
    axes[2, i+1].axis('off')

plt.suptitle('Simulating a Convolutional Layer: Input -> Convolution -> Activation -> Pooling', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key concepts:

1. **Convolution** slides a small filter over an image, producing a feature map that highlights specific patterns.
2. **Different filters detect different features**: horizontal edges, vertical edges, corners, textures, etc.
3. **Pooling** (max or average) reduces spatial dimensions while preserving the most important activations.
4. **A CNN stacks these operations**: Conv -> Activation -> Pool, repeated multiple times, then flattens and classifies.
5. In a real CNN, the **filter weights are learned** from data through backpropagation, rather than being hand-designed.

To train actual CNNs, you would use frameworks like PyTorch or TensorFlow, which handle automatic differentiation and GPU acceleration.